# Lab 08: Dynamic Web Pages with Selenium

*INFO 4617/5617 · Week 8 · Chapter 8, [Dynamic Web Pages with Selenium](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html)*

This week every class is live coding: one lab in three parts, one a day. Each step says what we're building and why. Its code cells hold a comment saying what to write, and we write the code together. Cells whose first comment starts with `# Given:` are already written: we read them together and run them. After each step, a **💬 Discuss** box holds questions to talk over; click it to open it. The lab itself isn't turned in. What you turn in is chapter 8's Recommended Exercises, in its companion notebook, due the Sunday after Friday's class.

We work on the chapter's practice site, [Quotes to Scrape](https://quotes.toscrape.com), which serves the same quotes as plain HTML, inside a script, through a hidden API, behind a ten-second delay, as an infinite scroll, and behind a login:
- **Part 1, Monday:** do you need a browser? We check with `requests` what a page sends, and call the hidden API behind another: one page by hand, then every page.
- **Part 2, Wednesday:** we drive Chrome from Python, headless. A page we know comes first; then we find, read, click, and wait.
- **Part 3, Friday:** we scroll a page to its end and hand it to BeautifulSoup, and log in by hand. After class, a Playwright script does the same work.

Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find. A part's time is its steps' minutes plus its microsprints'. What didn't fit a day's class is under that part's **After class** heading, with its minutes, for you to do before the next.

**Before class:** before Wednesday, finish [week 8's setup handout](../../handouts/week-08/README.md), so that Chrome has opened from chapter 8's notebook; Monday needs no browser. Open this notebook in the environment the handout used (`webdata`, or Anaconda's own), and have your email address ready for the User-Agent. Step 12, after Friday's class, also needs Playwright: "End of Part 2" says how to install it.

On Wednesday and Friday, start at that day's setup step.

# Part 1 — Monday: do you need a browser?

Ten quotes are on the screen at `quotes.toscrape.com/js/`. Today we check, with `requests` and no browser, whether they're in what the server sends. Then we find the hidden API that another page calls for its quotes, ask it for one page by hand, and let a given loop collect every page and save the table.

## Step 0 — Set up (4 minutes)

Put your email address in the User-Agent, and run the given cell. Today every request goes through `requests`, and says who sent it, as in week 2. The cell also imports `re` and `json` for Step 1, and `time` to time Step 3's loop and pause it.

In [ ]:
# Given: Part 1's setup. Put your email address in HEADERS, then run it
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}

<details>
<summary>💬 Discuss</summary>

- In week 2 we wrote a User-Agent that names us and how to reach us. On Wednesday, Chrome will send its own. Who will the sites think is visiting when Selenium drives it?

</details>

## Step 1 — What `requests` sees (12 minutes)

The chapter's first check is by hand: open <https://quotes.toscrape.com/js/>, right-click, and choose **View Page Source**. Ten quotes are on the screen, but there's no `<div class="quote">` in the source. Then turn JavaScript off (**Command Menu**, Control+Shift+P or Command+Shift+P, type `javascript`, **Disable JavaScript**) and reload: the quotes vanish ([Do You Need a Browser?](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#do-you-need-a-browser), Checks 1 and 2).

Now the same check in code. Week 6 counted 10 `div.quote` elements on the plain page, `quotes.toscrape.com/`: predict this page's count. Then look where View Source showed the quotes, a `<script>` that holds them as a JavaScript array named `data`. A given cell cuts it out with a **regular expression**, a pattern for text ([`re.search`](https://docs.python.org/3/library/re.html#re.search)), and parses it with [`json.loads()`](https://docs.python.org/3/library/json.html#json.loads), because it's written in JSON syntax.

In [ ]:
# js_response: fetch https://quotes.toscrape.com/js/ with HEADERS
# Print its status code and headers["Content-Type"]; then len(js_response.text), its characters
# Your code here

In [ ]:
# js_soup: parse js_response.text with BeautifulSoup; print how many "div.quote" it finds
# Your code here

In [ ]:
# Given: cut the array named data out of the page's script with a regular expression.
# "var data = (\[.*?\]);" captures from the "[" after "var data = " to the first "];",
# and re.DOTALL lets "." match line breaks too. Then parse what it captured as JSON
match = re.search(r"var data = (\[.*?\]);", js_response.text, re.DOTALL)
script_quotes = json.loads(match.group(1))

In [ ]:
# Print how many quotes script_quotes holds, then the first one's ["author"]["name"]
# Your code here

<details>
<summary>💬 Discuss</summary>

- `requests` got a 200 and found zero quotes, with no error. In week 6, IMDb answered with a 202 and zero characters. How would you notice either failure in a loop over a thousand pages?
- The quotes on `/js/` were in the response all along. When is cutting them out of a script better than starting a browser, and what could break it?
- Week 4's `json.loads()` parsed the array because it happens to be written in JSON syntax. What would happen if the page wrote its keys without quotation marks, as JavaScript allows?

</details>

## Step 2 — The hidden API, one page by hand (15 minutes)

The chapter's third check, by hand first: open <https://quotes.toscrape.com/scroll>, open developer tools' **Network** tab, click **Fetch/XHR**, and scroll. Each time you near the bottom, a request appears: `quotes?page=2`, `quotes?page=3`. Click one: its **Preview** is JSON, ten quotes and a `has_next` flag. This is a **hidden API**: nobody documents it, but everything you need to call it is in the Network tab ([Check 3: Watch the Network Tab](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#check-3-watch-the-network-tab)).

Ask for page 1 before any loop. [`params`](https://requests.readthedocs.io/en/stable/user/quickstart/#passing-parameters-in-urls) builds the `?page=1` for you. Look first: the [`Content-Type`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Content-Type) should say `application/json`. Then read it with `.json()`, and go down one level a cell. Then ask for page 2 the same way.

In [ ]:
# api_url: "https://quotes.toscrape.com/api/quotes"
# first_params: a dictionary whose "page" is 1
# Your code here

In [ ]:
# first_response: send it to api_url, with params=first_params and headers=HEADERS
# Print its .url; then its status code and headers["Content-Type"]
# Your code here

In [ ]:
# first_page: first_response.json(); print its keys
# Your code here

In [ ]:
# Print first_page's "page" and "has_next", and how many "quotes" it holds
# Your code here

In [ ]:
# first_quote: the first of first_page["quotes"]; print its keys
# Your code here

In [ ]:
# Print its "author"'s "name", one level down, and its "tags" (with an s)
# Your code here

In [ ]:
# Page 2 by hand: second_params, "page" 2; send it; second_page: its .json()
# Print second_page's "page" and "has_next", and how many "quotes" it holds
# Your code here

<details>
<summary>💬 Discuss</summary>

- We asked for page 1 and then page 2 before writing any loop. What did the second request tell us that the first couldn't?
- Typed as `"Page": 2`, with a capital P, the request still gets 200 and JSON, but page 1's quotes: the API ignores a parameter it doesn't know. Which printed value would catch it?
- In week 5, **Copy as cURL** showed every header the browser sent. If this API answered the browser with 200 and you with 403, what would you compare first?

</details>

## Step 3 — Every page: one past the end, then the whole loop (19 minutes)

Page 10 is the last. Before trusting a loop to stop, ask for page 11, one past the end: the chapter's probe in "Test one request".

The loop that follows `has_next` to the end is given: `get_all_quotes()` asks for page 1, 2, 3, … until `has_next` is false, a second apart. Read it together, and find Step 2's request in it. If a reply's status isn't 200, it stops and prints the start of the reply. It also takes a `tag`, the parameter the replies hint at with `"tag": null`; [`requests` leaves a parameter out](https://requests.readthedocs.io/en/stable/user/quickstart/#passing-parameters-in-urls) when its value is `None`. Time it. A given cell then makes a DataFrame with one row per quote, as the chapter's "Write the scraper" does ([Check 3](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#check-3-watch-the-network-tab)), and saves it.

In [ ]:
# past_params: "page" 11; send it to api_url; past_page: its .json()
# Print the status code, past_page["has_next"], and how many "quotes" it holds
# Your code here

In [ ]:
# Given: follow has_next from page 1 to the end, a second apart. Find Step 2's request in it.
# It reads api_url, from Step 2, and HEADERS, from Step 0. "while True:" repeats its indented
# lines until "return" leaves the function. A status other than 200 stops it, before .json()
def get_all_quotes(tag=None):
    quotes = []
    page = 1
    while True:
        params = {"page": page, "tag": tag}
        response = requests.get(api_url, params=params, headers=HEADERS)
        if response.status_code != 200:
            raise RuntimeError(f"Got {response.status_code}, not 200: {response.text[:200]}")
        data = response.json()
        quotes.extend(data["quotes"])
        if not data["has_next"]:
            return quotes
        page += 1
        time.sleep(1)

In [ ]:
# Time it: start = time.time(); all_quotes = get_all_quotes(); api_seconds = time.time() - start
# Print how many quotes came back, and round(api_seconds, 1)
# Your code here

In [ ]:
# Given: one row per quote, with its author's name, its text, and its tags joined by ", ".
# [... for q in all_quotes] is a list comprehension: it makes one dictionary for each quote q.
# Then save the table as Monday's checkpoint
api_quotes = pd.DataFrame([
    {"author": q["author"]["name"], "text": q["text"], "tags": ", ".join(q["tags"])}
    for q in all_quotes
])
api_quotes.to_csv("api_quotes.csv", index=False)

In [ ]:
# Print api_quotes.shape, and look at its first three rows
# Your code here

In [ ]:
# Given, only if the cells above stopped: remove the "# " (hash and space) from the next two lines
# (or select them, and press Ctrl+/ or Cmd+/), and run this cell. It loads the table the key's run
# saved on 2026-11-17
# data_base = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-08/data/"
# api_quotes = pd.read_csv(data_base + "api_quotes.csv")

<details>
<summary>💬 Discuss</summary>

- Page 11 answered 200, with an empty list and no error. What would a loop that stopped only on an error have done, and what does `get_all_quotes()` stop on instead?
- `get_all_quotes()` stops on any status but 200. Without that check, what would a 429's HTML page have done at `response.json()`, and would the error have named the 429? The chapter's loop stops with `response.raise_for_status()`.
- `api_quotes.csv` is a copy saved on one day. What can it not tell you that the live request can?

</details>

## End of Part 1

On Wednesday, Part 2 starts Chrome from Python: a page we know first, then find, click, type, and wait. It starts with its own setup step, so you don't need today's cells to have run. Before then, check that Chrome still opens from chapter 8's notebook (the setup handout's last step).

## After class: two microsprints (7 minutes)

These didn't fit Monday's 50 minutes. Do them before Wednesday. They use names from Steps 0–3: if you restarted the notebook, run Step 0's cell, and Steps 2 and 3's cells down to `get_all_quotes()`, first.

### ⏱ Microsprint from Step 1 (3 minutes, alone or in pairs)

**Do:** run Check 1 on another page of the practice site, <https://quotes.toscrape.com/scroll>. Fetch it, and ask whether its text holds `"var data"`, as `/js/` did, and whether it holds a name you can see on the page, `"Einstein"`. `in` asks whether one string is inside another, and answers `True` or `False`. If neither is there, the quotes reach the page some other way: Step 2 found it.
**Share:** both answers, and where you'd look for the quotes on a page of your own.

In [ ]:
# Microsprint: scroll_check: fetch https://quotes.toscrape.com/scroll with HEADERS
# Print whether "var data" is in scroll_check.text, then whether "Einstein" is
# Your code here

### ⏱ Microsprint from Step 3 (4 minutes, alone or in pairs)

**Do:** every reply carries `top_ten_tags`, the site's own count of its most-used tags, as a list of pairs; `dict()` turns it into a dictionary. Pick one tag, spelled as `top_ten` spells it (`"Humor"` returns no quotes, and no error), collect its quotes with `get_all_quotes(tag=...)`, and check the API's count against what came back.
**Share:** your tag, how many quotes came back, and whether that matches `top_ten_tags`.

In [ ]:
# Microsprint: top_ten = dict(first_page["top_ten_tags"]); print it, and pick a tag
# tagged: get_all_quotes(tag=...) for your tag; print len(tagged), then top_ten[your tag]
# Your code here

# Part 2 — Wednesday: drive a browser

When all three checks come up empty, you need a browser. Today we start Chrome from Python, headless, and open a page we know, to check that it sees what we see. Then we find elements on xkcd, read and click like a reader, and wait for a page that takes ten seconds.

## Step 4 — Set up again (5 minutes)

Part 2 starts fresh. Put your email address in the User-Agent, and run the three given cells. The second is step 1 of the chapter's [Before the First Browser](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#before-the-first-browser), which you ran in the setup handout. [Selenium Manager](https://www.selenium.dev/documentation/selenium_manager/#configuration) reads its settings from environment variables, so they're set before anything starts a browser: `SE_SKIP_DRIVER_IN_PATH` ignores stray drivers on your `PATH`, and `SE_AVOID_STATS` stops [its usage statistics](https://www.selenium.dev/documentation/selenium_manager/#opting-out-of-data-collection). The third builds Chrome's [`Options`](https://www.selenium.dev/documentation/webdriver/browsers/chrome/#options): `--headless=new` runs it with no window, and `--window-size=1280,800` lays pages out as a laptop window would ([Headless Mode](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#headless-mode)).

In [ ]:
# Given: Part 2's setup. Put your email address in HEADERS, then run it
import os
import time

import requests
from IPython.display import Image
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}

In [ ]:
# Given: Selenium Manager's settings, from the chapter's "Before the First Browser", step 1
import sys
from pathlib import Path

import selenium

manager = Path(sys.prefix) / ("Scripts/selenium-manager.exe" if os.name == "nt" else "bin/selenium-manager")
if "SE_MANAGER_PATH" not in os.environ and manager.is_file():
    os.environ["SE_MANAGER_PATH"] = str(manager)
os.environ["SE_SKIP_DRIVER_IN_PATH"] = "true"  # Ignore stray drivers on your PATH
os.environ["SE_AVOID_STATS"] = "true"          # Send no usage statistics
print(f"selenium {selenium.__version__}; Selenium Manager: {os.environ.get('SE_MANAGER_PATH', 'inside the selenium package')}")

In [ ]:
# Given: how every browser today starts. On your laptop, delete the "--headless=new" line to watch it
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1280,800")
# Key only: the key's server runs Chrome as root, which needs --no-sandbox ("Headless Mode").
# On a laptop LAB_CHROME_FLAGS isn't set, and these two lines do nothing
for flag in os.environ.get("LAB_CHROME_FLAGS", "").split():
    options.add_argument(flag)

<details>
<summary>💬 Discuss</summary>

- Selenium Manager sends usage statistics unless `SE_AVOID_STATS` is set. What does it learn about you, and would you have known without reading the chapter?
- The setup handout checked what Selenium Manager found before it started the first browser. Why check first, rather than read the error when `webdriver.Chrome()` fails?
- The key adds `--no-sandbox` because it runs as root on a server. Why does the chapter tell you to leave the sandbox on, on your own laptop, even when Chrome won't start?

</details>

## Step 5 — A page we know, then the one `requests` couldn't read (10 minutes)

Start Chrome with `webdriver.Chrome(options=options)`, and time it: a browser is a whole program, and starting one is slow ([Starting the Browser](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#starting-the-browser)). Run that cell once: each run starts another Chrome. Then a small first call, on a page whose answer you know: load the practice site's plain page, `quotes.toscrape.com/`, and print [`driver.title`](https://www.selenium.dev/documentation/webdriver/interactions/#get-title), the words in its `<title>`, which your browser's tab shows. Its `<h1>` heading says the same. Only then load `/js/`, where Step 1's `requests` found no quotes, and count them; if you get 0, run that cell again, and Step 8 says why. Last, [save a screenshot](https://www.selenium.dev/documentation/webdriver/interactions/windows/#takescreenshot): with no window, it's how you see what the browser saw.

In [ ]:
# Time the start: start = time.time(); driver = webdriver.Chrome(options=options)
# Print round(time.time() - start, 1), the seconds it took
# Your code here

In [ ]:
# Load https://quotes.toscrape.com/ with driver.get(), and print driver.title
# Your code here

In [ ]:
# heading: the page's first <h1>, with find_element(By.TAG_NAME, "h1"); print its .text
# Your code here

In [ ]:
# Load https://quotes.toscrape.com/js/; js_quotes: find_elements(By.CSS_SELECTOR, "div.quote")
# Print how many it found
# Your code here

In [ ]:
# Save a screenshot as "quotes-headless.png", and show it with Image(filename=..., width=640)
# Your code here

<details>
<summary>💬 Discuss</summary>

- The browser and Step 1's `requests` asked for the same `/js/`. Why did one find quotes and the other none?
- Headless Chrome names itself `HeadlessChrome` in its User-Agent, and every browser Selenium drives tells pages that `navigator.webdriver` is `true`. Week 5 called rewriting a User-Agent spoofing. Should a research scraper hide either one?
- What can the screenshot tell you that the count of quotes can't?

</details>

## Step 6 — Find elements (10 minutes)

[`find_element()`](https://www.selenium.dev/documentation/webdriver/elements/finders/) takes a **locator**: a strategy from `By` and a value. On xkcd's front page, count the navigation rows above and below the comic with `By.CLASS_NAME`, and their links with `By.CSS_SELECTOR`. A given cell then tries three more of the chapter's eight strategies ([The Eight Strategies](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#the-eight-strategies)). `find_element()` returns the first match; `find_elements()` returns a list, empty when there's none.

To search inside an element, call the method on the element, and start an XPath with a dot: without it, `//` searches the whole page ("One Element or Many").

In [ ]:
# Load https://xkcd.com; nav_rows: find_elements(By.CLASS_NAME, "comicNav"); print how many
# Your code here

In [ ]:
# nav_links: their links, find_elements(By.CSS_SELECTOR, "ul.comicNav a"); print how many
# Your code here

In [ ]:
# Given: three more strategies on the same page. [a.text for a in nav_links[:5]] is a list
# comprehension: the text of each of the first five links
print([a.text for a in nav_links[:5]])
# By link text, exact: the Random link's address
print(driver.find_element(By.LINK_TEXT, "Random").get_attribute("href"))
# By link text, partial: the first link whose text holds "Prev"
print(driver.find_element(By.PARTIAL_LINK_TEXT, "Prev").text)
# By XPath: up from the comic's <div> to the element that holds it, with "/.."
print(driver.find_element(By.XPATH, "//div[@id='comic']/..").get_attribute("id"))

In [ ]:
# first_row: nav_rows[0]. Inside it, count the links with By.XPATH ".//a",
# then with "//a" (no dot); print both counts
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** find the front page's comic image three different ways: `By.ID` for the comic's `<div>` and then `By.TAG_NAME` inside it, one `By.CSS_SELECTOR`, and one short `By.XPATH` that starts from the id. Check with `==` that all three found the same element, and print its `alt`, the comic's name.
**Share:** your three locators, whether `==` said `True`, and which one you'd trust after a redesign.

In [ ]:
# Microsprint: find the comic's <img> three ways: By.ID then By.TAG_NAME; By.CSS_SELECTOR; By.XPATH
# Compare the three with ==, and print the image's alt attribute
# Your code here

<details>
<summary>💬 Discuss</summary>

- Searching the first row with `//a` instead of `.//a` found every link on the page. What would a scraper with that bug have collected, and would it have raised an error?
- Typed as `"comicnav"`, lowercase, `find_elements()` returns an empty list with no error, while `find_element()` stops with `NoSuchElementException`. Which would you rather have in a loop over a thousand pages?
- In week 5, the Inspector's **Copy XPath** described an element in one click. Which of your three locators would survive xkcd adding a banner above the comic, and which would a counted-down path like `/html/body/div[2]/div[2]/img` not?

</details>

## Step 7 — Read what the browser shows, and click (9 minutes)

Selenium reads the page as the browser shows it. xkcd's `<div id="transcript">` is hidden, so its `.text` is empty, while [`get_attribute("textContent")`](https://www.selenium.dev/documentation/webdriver/elements/information/#fetching-attributes-or-properties) returns what the markup holds. Try both on comic 353, "Python". Then read the front page's comic: its `title` attribute holds the hover joke, and its `alt` the comic's name. Last, act like a reader: find the **Random** link with an XPath that matches part of its text with `contains()`, [`.click()`](https://www.selenium.dev/documentation/webdriver/elements/interactions/#click) it, and read the new comic ([Clicking](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#clicking)).

In [ ]:
# Load https://xkcd.com/353/, the comic "Python"; transcript: find_element(By.ID, "transcript")
# Print repr() of its .text: repr() shows an empty string as ''
# Your code here

In [ ]:
# Print the first 59 characters of transcript.get_attribute("textContent")
# Your code here

In [ ]:
# Load https://xkcd.com; comic_img: find_element(By.XPATH, "//div[@id='comic']//img")
# Print its "alt", then its "title", with get_attribute()
# Your code here

In [ ]:
# random_xpath: "//ul[@class='comicNav']//a[contains(text(),'Random')]"
# Find it with By.XPATH and .click() it; then time.sleep(1)
# Your code here

In [ ]:
# random_img: the comic image again; print driver.current_url, then the image's "title"
# Your code here

<details>
<summary>💬 Discuss</summary>

- Selenium said the transcript is empty; `textContent` found it. Which answer is right for a study of what readers see, and which for a study of what the page contains?
- `time.sleep(1)` after the click is a guess. What happens if the next comic takes three seconds to load?
- xkcd also publishes each comic's name, hover text and transcript as JSON, at `https://xkcd.com/353/info.0.json` ([its note on JSON](https://xkcd.com/json.html)). Did we need a browser for any of this?

</details>

## Step 8 — Wait for what you need (10 minutes)

`time.sleep()` guesses how long a page needs. An **explicit wait** asks instead: [`WebDriverWait`](https://www.selenium.dev/documentation/webdriver/waits/#explicit-waits) checks a condition every half second, returns as soon as it's true, and raises `TimeoutException` if it never is. That error's message is empty, so check the locator first. Conditions come from [`expected_conditions`](https://www.selenium.dev/selenium/docs/api/py/selenium_webdriver_support/selenium.webdriver.support.expected_conditions.html), imported as `EC` ([Waiting for Dynamic Content](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#waiting-for-dynamic-content)). `presence_of_element_located` takes a locator as one value, a **tuple**: a strategy and a value, in parentheses.

The practice page `/js-delayed/` waits ten seconds before its JavaScript adds the quotes. Look after a two-second sleep, then wait properly, and time it. Then quit: [`driver.quit()`](https://www.selenium.dev/documentation/webdriver/drivers/#quitting-sessions) closes every window and stops chromedriver. After it, a cell that uses `driver` stops with `MaxRetryError … Connection refused`: start a browser again.

In [ ]:
# start = time.time(); load https://quotes.toscrape.com/js-delayed/; time.sleep(2)
# Print how many elements have class "quote": find_elements(By.CLASS_NAME, "quote")
# Your code here

In [ ]:
# quote_locator = (By.CLASS_NAME, "quote"), a tuple. Wait for it, then print the seconds:
# WebDriverWait(driver, 15).until(EC.presence_of_element_located(quote_locator))
# Your code here

In [ ]:
# Count the elements with class "quote" again
# Your code here

In [ ]:
# Quit the browser: driver.quit()
# Your code here

<details>
<summary>💬 Discuss</summary>

- A scraper that slept two seconds would have saved an empty page for every quote, with no error. Where else today could a fixed sleep have failed silently?
- Step 5 timed Chrome's start, and Step 8 waited ten seconds for one page. Which of the two would a scraper of a thousand pages pay once, and which a thousand times?
- View Source on `/js-delayed/`: the quotes are in a script there too. Did this page need a browser at all?

</details>

## End of Part 2

On Friday, Part 3 scrolls a page to its end, hands it to BeautifulSoup, and logs in by hand, in a Chrome window. It starts with its own setup step, and loads Monday's table.

**Before Step 12**, after Friday's class: it needs Playwright's own browsers, about 320 MB, so download them at home, on a fast connection. In a terminal where the environment you use is active, run `python -m playwright install chromium`. If you set up Anaconda's own environment with the handout, rather than `webdata`, first add the library there: `conda install -c conda-forge playwright-python` ([Installing Playwright](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#installing-playwright)).

## After class: the rest of Part 2 (14 minutes)

These didn't fit Wednesday's 50 minutes: Step 8's microsprint (4 minutes), and the rest of Step 7, typing into a search form, with its microsprint (10 minutes). Step 8 quit the browser, so the microsprint starts a new one, and the typing uses it. If you restarted the notebook, run Step 4's three cells first, and set `quote_locator = (By.CLASS_NAME, "quote")`, as Step 8 did.

### ⏱ Microsprint from Step 8 (4 minutes, alone or in pairs)

**Do:** a short version of Additional Exercise 4. The given cell times `requests.get()` on `https://quotes.toscrape.com/js/`. Time a new browser the same way: from `webdriver.Chrome()` to the first quote on the same page. Then run both cells again: one run of each is one measurement.
**Share:** your times from both runs, and about how many times longer the browser took.

In [ ]:
# Given: how long requests.get() takes for https://quotes.toscrape.com/js/, and a check that it got the page
start = time.time()
timed_response = requests.get("https://quotes.toscrape.com/js/", headers=HEADERS)
requests_seconds = time.time() - start
if timed_response.status_code != 200:
    raise RuntimeError(f"Got {timed_response.status_code}, not 200: {timed_response.text[:200]}")
print(round(requests_seconds, 2))

In [ ]:
# Microsprint: start = time.time(); driver = webdriver.Chrome(options=options); load .../js/
# Wait up to 10 seconds for quote_locator; print round(time.time() - start, 2)
# Your code here

### Step 7, continued — Type into a search form (6 minutes)

Typing works like clicking: find the box, type into it with [`send_keys()`](https://www.selenium.dev/documentation/webdriver/elements/interactions/#send-keys), and press Enter with `Keys.RETURN` ([Typing and Searching](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#typing-and-searching)). Read a site's `robots.txt` before you send its forms anything (week 2). Wikipedia's search box would be the obvious example, but Wikipedia's `robots.txt` asks automated clients to stay out of `/w/` and `/wiki/Special:` pages, which is where its search sends you. So we type into a practice site's search: [Scrape This Site](https://www.scrapethissite.com/pages/forms/)'s hockey teams, one `<tr class="team">` row per team and season. When the address holds `q=`, the results have arrived. A given function, `search_teams()`, repeats the search for any query.

In [ ]:
# robots: fetch https://www.scrapethissite.com/robots.txt with HEADERS; print its status code and .text
# Your code here

In [ ]:
# Load https://www.scrapethissite.com/pages/forms/; search_box: find_element(By.NAME, "q")
# Your code here

In [ ]:
# Type and press Enter: search_box.send_keys("Colorado", Keys.RETURN). Wait until the address
# holds "q=": WebDriverWait(driver, 10).until(EC.url_contains("q=")). Print driver.current_url
# Your code here

In [ ]:
# teams: find_elements(By.CLASS_NAME, "team"); print how many, then the first one's .text
# Your code here

In [ ]:
# Given: the search above, for any query. It counts the teams listed on the page it lands on.
# It uses driver, from the microsprint above
def search_teams(query):
    driver.get("https://www.scrapethissite.com/pages/forms/")
    search_box = driver.find_element(By.NAME, "q")
    search_box.send_keys(query, Keys.RETURN)
    WebDriverWait(driver, 10).until(EC.url_contains("q="))
    teams = driver.find_elements(By.CLASS_NAME, "team")
    print(query, "|", len(teams), "teams on this page")

### ⏱ Microsprint from Step 7 (4 minutes, alone or in pairs)

**Do:** a short version of Additional Exercise 1. Run the given `search_teams()` on a team or a city of your own, and then on `"New York"`.
**Share:** your query, how many teams each listed, and what the "New York" page shows below its rows.

In [ ]:
# Microsprint: search_teams() with a query of your own, then with "New York"
# Your code here

<details>
<summary>💬 Discuss</summary>

- `"New York"` listed 25 teams on this page, and the page links to a second. How many New York teams are there, and how would a scraper that counted one page know it had missed some?
- A query that matches nothing lists 0 teams, and so does a misspelled one, or the class typed as `"teams"`. Would your code tell those apart?
- Week 2 read `robots.txt` and Terms of Service as rules for scrapers. Do they still apply when a real browser does the requesting and a script does the typing?

</details>

In [ ]:
# Quit the browser: driver.quit()
# Your code here

# Part 3 — Friday: scroll, and log in

On an infinite-scroll page, the quotes arrive as you scroll. Today we scroll once by hand, let a given loop scroll to the end, and hand the page to BeautifulSoup, to compare with Monday's API table. Then we log in to the practice site by hand. After class, a Playwright script does the same work, and waits on its own.

## Step 9 — Set up again (3 minutes)

Part 3 starts fresh. Put your email address in the User-Agent, and run the four given cells: Wednesday's imports, plus BeautifulSoup, pandas, and Step 8's `quote_locator`; Monday's table, `api_quotes`, loaded from the course's repository; and Selenium Manager's settings and `options`, as on Wednesday. Step 11 logs in by hand, in Chrome's window: on your laptop, delete the `--headless=new` line before you run the `options` cell.

In [ ]:
# Given: Part 3's setup. Put your email address in HEADERS, then run it
import os
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
quote_locator = (By.CLASS_NAME, "quote")

In [ ]:
# Given: Monday's table, api_quotes, saved by the key's run on 2026-11-17. Only Step 10's microsprint
# needs it. A 404 here means the file isn't on the course's repository yet: say so, and go on
data_base = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-08/data/"
api_quotes = pd.read_csv(data_base + "api_quotes.csv")
print(api_quotes.shape)

In [ ]:
# Given: Selenium Manager's settings, from the chapter's "Before the First Browser", step 1
import sys
from pathlib import Path

import selenium

manager = Path(sys.prefix) / ("Scripts/selenium-manager.exe" if os.name == "nt" else "bin/selenium-manager")
if "SE_MANAGER_PATH" not in os.environ and manager.is_file():
    os.environ["SE_MANAGER_PATH"] = str(manager)
os.environ["SE_SKIP_DRIVER_IN_PATH"] = "true"  # Ignore stray drivers on your PATH
os.environ["SE_AVOID_STATS"] = "true"          # Send no usage statistics
print(f"selenium {selenium.__version__}; Selenium Manager: {os.environ.get('SE_MANAGER_PATH', 'inside the selenium package')}")

In [ ]:
# Given: how every browser today starts. On your laptop, delete the "--headless=new" line: Step 11 needs a window
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1280,800")
# Key only: the key's server runs Chrome as root, which needs --no-sandbox ("Headless Mode").
# On a laptop LAB_CHROME_FLAGS isn't set, and these two lines do nothing
for flag in os.environ.get("LAB_CHROME_FLAGS", "").split():
    options.add_argument(flag)

<details>
<summary>💬 Discuss</summary>

- Monday's `api_quotes` came from a file saved on one day. If the site had changed a quote since, which table would today's comparison call wrong?

</details>

## Step 10 — Scroll to the end, then hand the page to BeautifulSoup (12 minutes)

On an infinite-scroll page, new content loads as you near the bottom. The chapter scrolls with [`execute_script()`](https://www.selenium.dev/documentation/webdriver/interactions/windows/#execute-script), which runs JavaScript in the page, and compares `document.body.scrollHeight`, the page's height in pixels, before and after each scroll: when the page stops growing, there's nothing more ([Scraping Infinite Scroll Pages](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#scraping-infinite-scroll-pages)). Scroll once by hand; how many quotes one scroll adds depends on timing. A given loop then scrolls to the end, at most 20 times, and times it. Rather than pause a fixed two seconds, it waits up to five for the page to grow. The API said 10 pages, so expect 100 quotes.

Then hand `driver.page_source`, the page as the browser built it, to BeautifulSoup ([Passing to BeautifulSoup](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#passing-to-beautifulsoup)): Selenium as the hands, BeautifulSoup as the eyes.

In [ ]:
# Start a browser with options, and load https://quotes.toscrape.com/scroll
# Your code here

In [ ]:
# Wait up to 10 seconds for quote_locator, as in Step 8; print how many "div.quote" it has
# Your code here

In [ ]:
# Print the page's height in pixels: driver.execute_script("return document.body.scrollHeight")
# Your code here

In [ ]:
# One scroll by hand: driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
# Then time.sleep(2), and print the count of "div.quote" and the height again
# Your code here

In [ ]:
# Given: scroll until the page stops growing, at most 20 times, and time it. Find the lines you just typed.
# It waits up to 5 seconds for the page to grow. "lambda d: ..." is a function in one line, which
# WebDriverWait calls until it's True. If it never is, the wait raises TimeoutException, which here
# means "nothing more": "except" catches it, and "break" leaves the loop
start = time.time()
reached_end = False
for scroll in range(1, 21):
    last_height = driver.execute_script("return document.body.scrollHeight")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    try:
        WebDriverWait(driver, 5).until(
            lambda d: d.execute_script("return document.body.scrollHeight") > last_height)
    except TimeoutException:
        reached_end = True
        break
scroll_seconds = round(time.time() - start, 1)
print("Scrolls:", scroll, "| reached the end:", reached_end, "| seconds:", scroll_seconds)

In [ ]:
# scroll_soup: driver.page_source, parsed with BeautifulSoup. Print len(driver.page_source),
# its characters, then how many "div.quote" scroll_soup finds
# Your code here

In [ ]:
# Given: save the page as the browser built it, the checkpoint for a student whose Chrome
# won't start. "with ... as f:" opens the file, and closes it after the indented line
with open("scroll-page.html", "w", encoding="utf-8") as f:
    f.write(driver.page_source)

In [ ]:
# Given, only if Chrome wouldn't start: remove the "# " (hash and space) from the next four lines
# (or select them, and press Ctrl+/ or Cmd+/), and run this cell. It loads the page the key's
# browser built on 2026-11-17
# scroll_response = requests.get(data_base + "scroll-page.html", headers=HEADERS)
# if scroll_response.status_code != 200:
#     raise RuntimeError(f"Got {scroll_response.status_code}, not 200: {scroll_response.text[:200]}")
# scroll_soup = BeautifulSoup(scroll_response.text, "html.parser")

In [ ]:
# Given: one row per quote on the scrolled page, with its author and its text
scroll_rows = []
for q in scroll_soup.find_all("div", class_="quote"):
    author = q.find("small", class_="author").get_text()
    text = q.find("span", class_="text").get_text()
    scroll_rows.append({"author": author, "text": text})
scroll_quotes = pd.DataFrame(scroll_rows)

In [ ]:
# Print scroll_quotes.shape; then quit the browser
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** check the scroll against Monday's API table: do the two give the same quotes? First print how many rows each table has. Then make a **set**, a collection with no repeats, of each table's `text`: `&` keeps what's in both sets, and `-` what's in the first only. A set drops repeats, so compare the sizes too.
**Share:** both tables' sizes, how many texts are in both, and how many in only one.

In [ ]:
# Microsprint: in_scroll: set(scroll_quotes["text"]); in_api: set(api_quotes["text"])
# Print len(scroll_quotes), len(api_quotes); then len(in_scroll & in_api), and each "-" both ways
# Your code here

<details>
<summary>💬 Discuss</summary>

- The browser and the API collected the same quotes. Which route would you defend in a methods section, and why?
- The loop takes five quiet seconds to mean "the end". On a slow connection, what would it have saved, and which printed value would tell you?
- Without its `return`, `execute_script("document.body.scrollHeight")` gives `None`, with no error. Which of your cells would have shown it, and what would the loop's `>` have done with `None`?
- `range(1, 21)` caps the scrolls at 20. On a feed with no end, how would you choose the cap, and what does week 2's proportionality question say about it?

</details>

## Step 11 — Log in by hand (10 minutes)

Quotes to Scrape has a practice login that accepts any username and password. Logged in, each quote shows a **(Goodreads page)** link that logged-out visitors don't see, so counting those links tells you whether you're logged in ([Pages Behind a Login](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#pages-behind-a-login)). Log in the chapter's first way: by hand, in Chrome's window, while the cell waits for you with `input()` ([Logging In by Hand](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#logging-in-by-hand)). Your password goes from your keyboard to the site, never into the notebook. This needs a window: if Step 9's `options` cell still has `--headless=new`, delete that line and run the cell again. Never print a cookie's value; its *name* is safe.

In [ ]:
# Start a new browser with options, and load https://quotes.toscrape.com/
# logged_out: find_elements(By.LINK_TEXT, "(Goodreads page)"); print how many
# Your code here

In [ ]:
# Load https://quotes.toscrape.com/login; then hold the cell while you log in, in Chrome's window:
# input("Log in in the Chrome window, then press Enter here")
# Your code here

In [ ]:
# Load https://quotes.toscrape.com/ again; logged_in: the Goodreads links again; print how many
# Your code here

In [ ]:
# Given: the names of the cookies the site set, never their values.
# [cookie["name"] for cookie in ...] is a list comprehension: the name of each cookie
print([cookie["name"] for cookie in driver.get_cookies()])

In [ ]:
# Quit the browser: driver.quit()
# Your code here

<details>
<summary>💬 Discuss</summary>

- Typed as `"Goodreads page"`, without the parentheses, the count is 0 logged in or out, with no error. How do you know that the first 0 meant "logged out"?
- Technically, logging in set one cookie. What else changed: for you, for the site, and for the people whose posts you'd see?
- A job that runs on a schedule can't wait for you to press Enter. The chapter's "Passwords and Other Credentials" reads the password from an environment variable or `getpass()` instead. Why not type it into a cell?
- Week 3 put X's enterprise API at $42,000 a month and up. Is a logged-in browser the cheaper route, and who pays the difference?

</details>

## Step 13 — What did we learn? (3 minutes)

In a few sentences, as comments or in a Markdown cell:

- Of the checks in Steps 1–3, which found the quotes without a browser, and on which pages? How many requests and seconds did the API take in Step 3, against the scrolls and seconds of Step 10's loop? Both include their pauses. One line can answer it: "API: … requests in … s; scroll: … scrolls in … s."
- What did a browser cost to start in Step 5, and how long did Step 8 wait for one page? For which page this week was a browser the only way in?
- In Step 11, what told you that you were logged in, and what did logging in cost besides the code?
- This week's cells assumed every request and every wait would work. Which assumption would you guard first on a site of your own, and how does the chapter guard it?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Exercise Step 1, fetch your page statically and count your element: Step 1's `requests` and BeautifulSoup check, with `len(js_response.text)`, and week 2's `robots.txt`, which Step 7's search form read first (after class).
- Exercise Step 2, start a headless browser: Step 4's `options`, and Step 5's `webdriver.Chrome()` and `driver.get()`.
- Exercise Step 3, wait, then measure again: Step 8's `WebDriverWait`, and Step 10's `len(driver.page_source)` and BeautifulSoup.
- Exercise Step 4, load more: Step 10's scroll by hand and its given loop, or Step 7's `.click()` for a **Load More** button.
- Exercise Step 5, tabulate and clean up: Step 3's DataFrame from a list of dictionaries, and `driver.quit()` (Steps 8 and 10).
- Exercise Step 6, interpret: Steps 1–3's checks, and the timings of Steps 3, 5, 8 and 10.

In [ ]:
# Your answers here, as comments


## After class: the rest of Part 3 (13 minutes)

These didn't fit Friday's 35 minutes: Step 11's microsprint (5 minutes), on paper, and Step 12 (8 minutes), a Playwright script, which needs the install in "End of Part 2". Step 12 uses `sys`, from Step 9's Selenium Manager cell: if you restarted the notebook, run that cell first.

### ⏱ Microsprint from Step 11 (5 minutes, alone or in pairs)

**Do:** a short version of Additional Exercise 6, on paper. Choose a site where you have an account and data you'd like to study, and read its `robots.txt` first. Without collecting anything else, list the routes to the data: an API and its price, a researcher program, data donation, asking the site, and a logged-in browser. For each, note who carries the risk: you, the platform, or the people in the data.
**Share:** your site, what its `robots.txt` says, the route you'd take, and one risk that route puts on someone other than you.

In [ ]:
# Microsprint: answer as comments. Your site, its robots.txt, each route with its cost and who carries the risk, and the route you'd take


## Step 12 — The same with a Playwright script (8 minutes)

[Playwright](https://playwright.dev/python/docs/library) drives a browser too: it installs its own browsers, and its [**locators**](https://playwright.dev/python/docs/locators) wait on their own when you act on an element or read from it. `count()` doesn't: it reports the page as it stands. Its simple *sync* API won't run inside a notebook, because Jupyter already runs an event loop of its own ([Why Not in the Notebook?](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#why-not-in-the-notebook)). So a cell that starts with `%%writefile` saves a script, and `!"{sys.executable}" file.py` runs it with the notebook's Python, as a terminal would; the quotation marks keep a folder name with a space in it in one piece.

A small first script is given: it opens the plain page and prints [`page.title()`](https://playwright.dev/python/docs/api/class-page#page-title). Predict it from Step 5. Copy it, point it at `/js/`, and count the quotes. Last, a given script catches the API reply that scrolling triggers ([Catching the JSON Behind a Page](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-08-dynamic-pages.html#catching-the-json-behind-a-page)): Step 2's request.

In [ ]:
%%writefile hello_pw.py
# Given: a first Playwright script. It opens the practice site's plain page, and prints its title
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page()
    page.goto("https://quotes.toscrape.com/")
    print(page.title())
    browser.close()

In [ ]:
# Run it with this notebook's Python, as a separate program: !"{sys.executable}" hello_pw.py
# Your code here

In [ ]:
%%writefile js_pw.py
# Copy everything after the first line of hello_pw.py's cell here; change the address to
# .../js/, and print page.locator("div.quote").count() instead of the title
# Your code here

In [ ]:
# Run js_pw.py the same way
# Your code here

In [ ]:
%%writefile scroll_api.py
# Given: open /scroll and print the first quote's author (the locator waits for it). Then scroll,
# and catch the reply to "/api/quotes?page=2" as the page receives it. "lambda r: ..." is a
# function in one line: it is True for the response r whose address holds that text
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page()
    page.goto("https://quotes.toscrape.com/scroll")
    print(page.locator("div.quote small.author").first.inner_text())
    with page.expect_response(lambda r: "/api/quotes?page=2" in r.url) as info:
        page.mouse.wheel(0, 20000)
    data = info.value.json()
    print(data["page"], len(data["quotes"]))
    browser.close()

In [ ]:
# Run scroll_api.py the same way
# Your code here

<details>
<summary>💬 Discuss</summary>

- Playwright's locator waited for the first quote without a `WebDriverWait`. Which of this week's Selenium lines did the script not need?
- Point `js_pw.py` at `/js-delayed/` and run it again. Why does `count()` print 0 there, and which line of `scroll_api.py` would have waited?
- An AI agent connected through Playwright MCP could have collected these quotes for you. If it had, what could your methods section cite, compared with a script it wrote for you to read and rerun?
- Selenium drives Chrome, Firefox, Edge, or Safari as you have them installed; Playwright brings its own copies, pinned to its version. Which would you rather explain in a replication package, and why?

</details>

## Friday: from the lab to the textbook (15 minutes)

What did the lab turn up that chapter 8 could do better? Some places to look:

- **Outputs that changed.** xkcd's front page and the random comic change; did the chapter's xkcd numbers (2 navigation rows, 10 links) still hold when you ran Step 6? How long did Step 5's browser take to start on your laptop?
- **A cell that broke**, on your laptop and not on the instructor's screen: Selenium Manager, a missing driver, a window that wouldn't open, Playwright's browsers. The chapter's "When Selenium Manager Fails" and "Common Issues to Debug" are where a fix would go.
- **A sentence that confused you.** The chapter's examples run on whatever page the browser happens to be on: which page is it on when "Scrolling" presses Page Down, or when "Passing to BeautifulSoup" counts links?
- **A microsprint that would make a good exercise**, such as checking the API's `top_ten_tags` against a tagged loop, or the three locators for one element.

See [`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) for how to propose a revision.